# Κεφάλαιο 9: Το επίπεδο διανυσματικών αναπαραστάσεων

> «Θα καταλάβεις μια λέξη από τις παρέες της.» — **John Rupert Firth**, γλωσσολόγος

---

## Τι θα μάθεις

- Γιατί δεν αρκούν από μόνα τους τα ID των token (μονάδων κειμένου) και ποιο πρόβλημα λύνουν στην πράξη οι διανυσματικές αναπαραστάσεις (embedding)
- Πώς να φτιάξεις από το μηδέν διανυσματικές αναπαραστάσεις των token χρησιμοποιώντας πίνακες αναζήτησης
- Γιατί έχει σημασία η πληροφορία θέσης και πώς να την προσθέσεις
- Πώς να συνδυάσεις τις διανυσματικές αναπαραστάσεις των token και των θέσεών τους για να σχηματίσεις πλήρεις αναπαραστάσεις εισόδου
- Πώς να εξερευνήσεις τις διανυσματικές αναπαραστάσεις πραγματικών μοντέλων και να ανακαλύψεις σημασιολογικές σχέσεις
- Τι να προσέξεις στην πράξη κατά την αρχικοποίηση και την υλοποίηση

---

## Προετοιμασία

Πρώτα, εγκατέστησε τα απαραίτητα πακέτα:

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch09.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch09.ipynb). Κρατάμε τα αγγλικά κείμενα των παραδειγμάτων ώστε οι είσοδοι, τα ID και οι ετικέτες τους να συμφωνούν.


In [ ]:
# Εγκατάσταση των απαραίτητων πακέτων
!pip install -q torch==2.15.0 transformers==5.23.0

In [ ]:
# ===== ΕΙΣΑΓΩΓΕΣ =====
import torch                     # PyTorch: πράξεις με τανυστές και νευρωνικά δίκτυα
import torch.nn as nn            # Δομικές μονάδες νευρωνικών δικτύων (επίπεδα κτλ.)
import torch.nn.functional as F  # Μαθηματικές συναρτήσεις (cosine_similarity κτλ.)
from transformers import AutoModel, AutoTokenizer  # Προεκπαιδευμένα μοντέλα

# Σύντομη υπενθύμιση για τους τανυστές:
# - Το torch.tensor([1,2,3]) δημιουργεί έναν μονοδιάστατο τανυστή (σαν λίστα)
# - Το torch.randn(3, 4) δημιουργεί έναν τανυστή 3×4 με τυχαίους αριθμούς
# - Με το tensor[0] επιλέγεις το στοιχείο με δείκτη 0 στην πρώτη διάσταση
# - Το tensor.shape σου δείχνει το σχήμα του τανυστή

## 1. Γιατί δεν αρκεί να χρησιμοποιήσεις τα ID των token;

Τα ID των token είναι αυθαίρετοι ακέραιοι που από μόνοι τους δεν δηλώνουν τίποτα για τη σημασία των token. Δες ποιο είναι το πρόβλημα:

In [ ]:
# Τα ID των token είναι απλώς ακέραιοι — δεν εκφράζουν σημασιολογικές σχέσεις
# (πραγματικά ID του GPT-2· το κενό στην αρχή έχει σημασία: " cat" αντί για "cat")
token_ids = {
    " cat": 3797,
    " kitten": 42143,
    " dog": 3290,
    " car": 1097
}

print("ID των token:")
for word, id in token_ids.items():
    print(f"  '{word}' → {id}")

# Το πρόβλημα: αριθμητικά, το "cat" είναι πιο κοντά στο "dog" παρά στο "kitten"!
print(f"\nΑπόσταση από το 'cat' στο 'dog': {abs(3797 - 3290)}")
print(f"Απόσταση από το 'cat' στο 'kitten': {abs(3797 - 42143)}")
print("\nΩς προς τη σημασία τους, όμως, το 'cat' και το 'kitten' μοιάζουν περισσότερο!")

### Η λύση: Διανυσματικές αναπαραστάσεις

Μετέτρεψε το ID κάθε token σε ένα πυκνό διάνυσμα που αποτυπώνει τη σημασία του token.

**Τι είναι το `nn.Embedding`;**
- Ένας πίνακας αναζήτησης με `vocab_size` γραμμές και `embed_dim` στήλες
- Κάθε γραμμή είναι ένα διάνυσμα που αναπαριστά ένα token
- Είσοδος: ID ενός token (ακέραιος) → Έξοδος: η αντίστοιχη γραμμή (διάνυσμα)
- Σκέψου τον σαν λεξικό: `{0: [0.1, 0.2, ...], 1: [0.5, -0.1, ...], ...}`

In [ ]:
# ===== Το πρόβλημα: Τα αναγνωριστικά των token (μονάδων κειμένου) =====
token_ids = torch.tensor([3797, 42143, 3290])  # " cat", " kitten", " dog"
print(f"Σχήμα του τανυστή αναγνωριστικών token: {token_ids.shape}")
print(f"Απλώς ακέραιοι αριθμοί: {token_ids}")

# ===== Η λύση: Διανυσματικές αναπαραστάσεις (embedding) =====
vocab_size = 50257  # Μέγεθος λεξιλογίου του GPT-2
embed_dim = 768     # Διάσταση της διανυσματικής αναπαράστασης του GPT-2

# Δημιούργησε ένα επίπεδο διανυσματικών αναπαραστάσεων (είναι ένας πίνακας αναζήτησης!)
embedding = nn.Embedding(vocab_size, embed_dim)

# Βρες τα διανύσματα που αντιστοιχούν στα αναγνωριστικά των token
token_vectors = embedding(token_ids)
print(f"\nΣχήμα του τανυστή διανυσματικών αναπαραστάσεων των token: {token_vectors.shape}")
print(f"Διάνυσμα του πρώτου token (πρώτες 10 συνιστώσες): {token_vectors[0, :10]}")

# Τώρα κάθε token είναι ένα διάνυσμα 768 διαστάσεων που μπορεί να αποτυπώσει νόημα!

## 2. Διανυσματικές αναπαραστάσεις των token: Ο πίνακας αναζήτησης

Φτιάξε διανυσματικές αναπαραστάσεις των token από την αρχή για να καταλάβεις πώς λειτουργεί ο μηχανισμός.

### Βήμα 1: Δημιούργησε τον πίνακα διανυσματικών αναπαραστάσεων

In [ ]:
vocab_size = 50257  # Μέγεθος λεξιλογίου του GPT-2
embed_dim = 768     # Διάσταση της διανυσματικής αναπαράστασης

# Δημιούργησε τον πίνακα διανυσματικών αναπαραστάσεων: μία γραμμή για κάθε token
embedding_matrix = torch.randn(vocab_size, embed_dim)

print(f"Σχήμα του πίνακα διανυσματικών αναπαραστάσεων: {embedding_matrix.shape}")
print(f"\nΔιανυσματική αναπαράσταση του token 3797 (πρώτες 10 συνιστώσες): {embedding_matrix[3797, :10]}")

### Βήμα 2: Βρες τα διανύσματα για πολλά token

In [ ]:
token_ids = torch.tensor([464, 3797, 3332])  # "The cat sat"

# Χειροκίνητη αναζήτηση (αυτό κάνουν εσωτερικά τα επίπεδα διανυσματικών αναπαραστάσεων)
embeddings = embedding_matrix[token_ids]

print(f"Σχήμα εισόδου: {token_ids.shape}")       # torch.Size([3])
print(f"Σχήμα εξόδου: {embeddings.shape}")     # torch.Size([3, 768])

# Κάθε αναγνωριστικό token → το αντίστοιχο διάνυσμα 768 διαστάσεων
print(f"\nΔιανυσματική αναπαράσταση του token 464 (πρώτες 5 συνιστώσες): {embeddings[0, :5]}")
print(f"Διανυσματική αναπαράσταση του token 3797 (πρώτες 5 συνιστώσες): {embeddings[1, :5]}")
print(f"Διανυσματική αναπαράσταση του token 3332 (πρώτες 5 συνιστώσες): {embeddings[2, :5]}")

### Βήμα 3: Επεξεργάσου παρτίδες

In [ ]:
# Παρτίδα με 2 ακολουθίες των 4 token η καθεμία
token_ids_batch = torch.tensor([
    [464, 3797, 3332, 319],    # Ακολουθία 1: "The cat sat on"
    [40, 481, 1037, 345]       # Ακολουθία 2: "I will help you"
])

print(f"Σχήμα παρτίδας: {token_ids_batch.shape}")  # torch.Size([2, 4])

# Ανάκτησε τις διανυσματικές αναπαραστάσεις για ολόκληρη την παρτίδα
embeddings_batch = embedding_matrix[token_ids_batch]

print(f"Σχήμα διανυσματικών αναπαραστάσεων: {embeddings_batch.shape}")  # torch.Size([2, 4, 768])
print("\nΑλλαγή σχήματος: (μέγεθος παρτίδας, μήκος ακολουθίας) → (μέγεθος παρτίδας, μήκος ακολουθίας, διάσταση αναπαράστασης)")

### Βήμα 4: Χρησιμοποίησε το nn.Embedding του PyTorch

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Επίπεδο διανυσματικών αναπαραστάσεων token: μετατρέπει τα αναγνωριστικά των token σε πυκνά διανύσματα.
    
    Αυτό κάνει το επίπεδο 'wte' (διανυσματικές αναπαραστάσεις token) του GPT-2.
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Δημιούργησε τον πίνακα διανυσματικών αναπαραστάσεων ως εκπαιδεύσιμη παράμετρο
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Αρχικοποίησε τον πίνακα με μικρές τυχαίες τιμές (όπως στο GPT-2)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids):
        """
        Ορίσματα:
            token_ids: τανυστής αναγνωριστικών token με σχήμα (μέγεθος παρτίδας, μήκος ακολουθίας)
        
        Επιστρέφει:
            embeddings: τανυστής διανυσμάτων token με σχήμα (μέγεθος παρτίδας, μήκος ακολουθίας, διάσταση αναπαράστασης)
        """
        return self.embedding(token_ids)

# Δημιούργησε το επίπεδο διανυσματικών αναπαραστάσεων token
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)

# Πάρε τις διανυσματικές αναπαραστάσεις για μία παρτίδα
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Σχήμα: (1, 4)
embeddings = token_embed(token_ids)

print(f"Σχήμα εισόδου: {token_ids.shape}")        # torch.Size([1, 4])
print(f"Σχήμα εξόδου: {embeddings.shape}")      # torch.Size([1, 4, 768])
print(f"\nΔιανυσματική αναπαράσταση του πρώτου token (πρώτες 5 συνιστώσες): {embeddings[0, 0, :5]}")

## 3. Διανυσματικές αναπαραστάσεις θέσης: Πώς μαθαίνει το μοντέλο τη θέση

Η διανυσματική αναπαράσταση ενός token δεν περιέχει πληροφορία για τη θέση του. Δες το πρόβλημα:

In [ ]:
# Δύο ακολουθίες με τα ίδια token σε διαφορετική σειρά
token_ids_1 = torch.tensor([[464, 3797, 3332]])  # "The cat sat"
token_ids_2 = torch.tensor([[3332, 3797, 464]])  # "sat cat The"

# Πάρε τις διανυσματικές αναπαραστάσεις των token
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)
embeddings_1 = token_embed(token_ids_1)
embeddings_2 = token_embed(token_ids_2)

print(f"Σχήμα διανυσματικών αναπαραστάσεων της ακολουθίας 1: {embeddings_1.shape}")
print(f"Σχήμα διανυσματικών αναπαραστάσεων της ακολουθίας 2: {embeddings_2.shape}")

# Οι δύο τανυστές διαφέρουν...
print(f"\nΕίναι ίδιες οι διανυσματικές αναπαραστάσεις; {torch.equal(embeddings_1, embeddings_2)}")

# Στο δεύτερο αποτέλεσμα, αντίστρεψε τη σειρά των διανυσμάτων στην ακολουθία, όχι τη σειρά των συντεταγμένων τους.
assert torch.equal(embeddings_1, embeddings_2.flip(1))
print("Τα ίδια διανύσματα token σε αντίστροφη σειρά: η ανάκτηση από τον πίνακα δεν προσθέτει πληροφορία θέσης.")

### Υλοποίηση εκπαιδεύσιμων διανυσματικών αναπαραστάσεων θέσης

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Διανυσματικές αναπαραστάσεις θέσης που μαθαίνονται κατά την εκπαίδευση: ένα διάνυσμα ανά θέση.
    
    Το GPT-2 χρησιμοποιεί αυτή την προσέγγιση ('wpe' — διανυσματικές αναπαραστάσεις θέσης λέξεων).
    """
    def __init__(self, max_seq_len, embed_dim):
        """
        Ορίσματα:
            max_seq_len: Μέγιστο μήκος ακολουθίας (π.χ. 1024 για το GPT-2)
            embed_dim: Διάσταση αναπαράστασης (πρέπει να ταιριάζει με τη διάσταση των αναπαραστάσεων token)
        """
        super().__init__()
        # Δημιούργησε τον πίνακα διανυσματικών αναπαραστάσεων θέσης: (max_seq_len, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Αρχικοποίησε τα βάρη με μικρές τυχαίες τιμές (όπως στο GPT-2)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Ορίσματα:
            token_ids: τανυστής με αναγνωριστικά token και σχήμα (παρτίδα, μήκος ακολουθίας)
        
        Επιστρέφει:
            pos_embeddings: τανυστής με διανύσματα θέσης και σχήμα (παρτίδα, μήκος ακολουθίας, embed_dim)
        """
        batch_size, seq_len = token_ids.shape
        
        # Έλεγξε το μήκος της ακολουθίας
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Το μήκος ακολουθίας {seq_len} υπερβαίνει το όριο max_seq_len {self.max_seq_len}"
            )
        
        # Δημιούργησε τους δείκτες θέσης: [0, 1, 2, ..., seq_len-1]
        position_ids = torch.arange(
            seq_len,
            device=token_ids.device  # Χρησιμοποίησε την ίδια συσκευή (CPU/GPU) με την είσοδο
        )
        
        # Επέκτεινε τους δείκτες για όλη την παρτίδα: (seq_len,) → (batch, seq_len)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        
        # Ανάκτησε τις διανυσματικές αναπαραστάσεις θέσης
        pos_embeddings = self.pos_embed(position_ids)
        
        return pos_embeddings

# Δημιούργησε το επίπεδο διανυσματικών αναπαραστάσεων θέσης
pos_embed = PositionalEmbedding(max_seq_len=1024, embed_dim=768)

# Παράδειγμα: ακολουθία μήκους 4
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Σχήμα: (1, 4)
pos_embeddings = pos_embed(token_ids)

print(f"Σχήμα εισόδου: {token_ids.shape}")           # torch.Size([1, 4])
print(f"Σχήμα αναπαραστάσεων θέσης: {pos_embeddings.shape}")  # torch.Size([1, 4, 768])

# Σε κάθε θέση αντιστοιχεί ένα ξεχωριστό διάνυσμα που μαθαίνεται κατά την εκπαίδευση
print(f"\nΑναπαράσταση θέσης 0 (πρώτες 5 συνιστώσες): {pos_embeddings[0, 0, :5]}")
print(f"Αναπαράσταση θέσης 1 (πρώτες 5 συνιστώσες): {pos_embeddings[0, 1, :5]}")
print(f"Αναπαράσταση θέσης 2 (πρώτες 5 συνιστώσες): {pos_embeddings[0, 2, :5]}")
print(f"Αναπαράσταση θέσης 3 (πρώτες 5 συνιστώσες): {pos_embeddings[0, 3, :5]}")

## 4. Συνδυασμός διανυσματικών αναπαραστάσεων token και θέσης

### Φτιάξε την πλήρη κλάση GPT2Embeddings

In [ ]:
class GPT2Embeddings(nn.Module):
    """
    Πλήρες επίπεδο διανυσματικών αναπαραστάσεων του GPT-2: αναπαραστάσεις token + θέσης.
    
    Αντιστοιχεί στα επίπεδα 'wte' + 'wpe' του GPT-2.
    """
    def __init__(self, vocab_size, max_seq_len, embed_dim):
        """
        Ορίσματα:
            vocab_size: Μέγεθος λεξιλογίου (50257 για το GPT-2)
            max_seq_len: Μέγιστο μήκος ακολουθίας (1024 για το GPT-2)
            embed_dim: Διάσταση διανυσματικής αναπαράστασης (768 για το GPT-2 Small)
        """
        super().__init__()
        
        # Διανυσματικές αναπαραστάσεις token: vocab_size × embed_dim
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Διανυσματικές αναπαραστάσεις θέσης: max_seq_len × embed_dim
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Αρχικοποίησε και τους δύο πίνακες με την τυπική αρχικοποίηση του GPT-2
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Ορίσματα:
            token_ids: τανυστής με αναγνωριστικά token και σχήμα (παρτίδα, μήκος ακολουθίας)
        
        Επιστρέφει:
            embeddings: τανυστής με τις συνδυασμένες αναπαραστάσεις και σχήμα (παρτίδα, μήκος ακολουθίας, embed_dim)
        """
        batch_size, seq_len = token_ids.shape
        
        # Έλεγξε το μήκος της ακολουθίας
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Το μήκος ακολουθίας {seq_len} υπερβαίνει το όριο max_seq_len {self.max_seq_len}"
            )
        
        # ===== Διανυσματικές αναπαραστάσεις token =====
        token_embeddings = self.token_embed(token_ids)
        
        # ===== Διανυσματικές αναπαραστάσεις θέσης =====
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeddings = self.pos_embed(position_ids)
        
        # ===== Συνδύασέ τες με πρόσθεση =====
        embeddings = token_embeddings + position_embeddings
        
        return embeddings

# Δημιούργησε το επίπεδο διανυσματικών αναπαραστάσεων του GPT-2 Small
gpt2_embed = GPT2Embeddings(
    vocab_size=50257,
    max_seq_len=1024,
    embed_dim=768
)

# Παράδειγμα: βρες τις αναπαραστάσεις μιας παρτίδας ακολουθιών (ΚΑΙ ΟΙ ΔΥΟ πρέπει να έχουν το ίδιο μήκος!)
# Το GPT-2 δεν έχει ειδικό token συμπλήρωσης: το αναγνωριστικό 0 αντιστοιχεί στο '!', όχι σε συμπλήρωση.
# Συνήθως (Κεφάλαιο 12) η συμπλήρωση γίνεται με το token τέλους κειμένου, με αναγνωριστικό 50256.
token_ids = torch.tensor([
    [464, 3797, 3332, 319, 50256, 50256],  # "The cat sat on" + συμπλήρωση
    [40, 481, 1037, 345, 50256, 50256]     # "I will help you" + συμπλήρωση
])

embeddings = gpt2_embed(token_ids)

print(f"Σχήμα εισόδου: {token_ids.shape}")         # torch.Size([2, 6])
print(f"Σχήμα εξόδου: {embeddings.shape}")       # torch.Size([2, 6, 768])
print(f"Κάθε token αναπαρίσταται πλέον σε {embeddings.shape[-1]} διαστάσεις")

print(f"\nΠρώτη ακολουθία, πρώτο token (πρώτες 10 συνιστώσες):")
print(embeddings[0, 0, :10])

### Σύνδεση με το Κεφάλαιο 8: Η διαδικασία από την αρχή ως το τέλος

In [ ]:
from transformers import AutoTokenizer

# ===== Βήμα 1: Χώρισε το κείμενο σε token (Κεφάλαιο 8) =====
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text, return_tensors="pt")

print("Βήμα 1: Διαχωρισμός σε token")
print(f"Κείμενο: {text}")
print(f"Αναγνωριστικά token: {token_ids}")
print(f"Σχήμα: {token_ids.shape}\n")

# ===== Βήμα 2: Υπολόγισε τις διανυσματικές αναπαραστάσεις (Κεφάλαιο 9) =====
gpt2_embed = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = gpt2_embed(token_ids)

print("Βήμα 2: Διανυσματική αναπαράσταση")
print(f"Σχήμα αναπαραστάσεων: {embeddings.shape}")
print(f"Αναπαράσταση του πρώτου token (πρώτες 10 συνιστώσες): {embeddings[0, 0, :10]}\n")

print("Βήμα 3: Στη συνέχεια — Επίπεδα προσοχής (Κεφάλαιο 10)")
print(f"Ο τανυστής αναπαραστάσεων με σχήμα {embeddings.shape} θα περάσει στην αυτοπροσοχή,")
print("όπου η αναπαράσταση κάθε token εμπλουτίζεται με πληροφορίες από τα συμφραζόμενά του!")

## 5. Εξερεύνησε τις διανυσματικές αναπαραστάσεις του GPT-2

Φόρτωσε ένα πραγματικό προεκπαιδευμένο μοντέλο GPT-2 και εξερεύνησε τι έμαθε:

In [ ]:
# Φόρτωσε το GPT-2 Small
model = AutoModel.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Δες τις διανυσματικές αναπαραστάσεις των token
token_embeddings = model.wte.weight  # Διανυσματικές αναπαραστάσεις των token (μονάδων κειμένου)
print(f"Σχήμα των διανυσματικών αναπαραστάσεων των token: {token_embeddings.shape}")
# torch.Size([50257, 768]) — ένα διάνυσμα 768 διαστάσεων ανά token

# Δες τις διανυσματικές αναπαραστάσεις των θέσεων
position_embeddings = model.wpe.weight  # Διανυσματικές αναπαραστάσεις των θέσεων
print(f"Σχήμα των διανυσματικών αναπαραστάσεων των θέσεων: {position_embeddings.shape}")
# torch.Size([1024, 768]) — ένα διάνυσμα 768 διαστάσεων ανά θέση

### Βρες παρόμοιες λέξεις

**Τι είναι η ομοιότητα συνημιτόνου;**
Μετρά πόσο μοιάζουν δύο διανύσματα με βάση τη γωνία που σχηματίζουν:
- **1.0** = δείχνουν προς την ίδια κατεύθυνση (μέγιστη ομοιότητα συνημιτόνου)
- **0.0** = είναι κάθετα μεταξύ τους (μηδενική ομοιότητα συνημιτόνου)
- **-1.0** = δείχνουν προς αντίθετες κατευθύνσεις (χωρίς αυτό να σημαίνει απαραίτητα αντίθετες έννοιες)

Σκέψου το έτσι: «Δείχνουν αυτά τα δύο βέλη προς την ίδια κατεύθυνση;»

In [ ]:
def find_similar_tokens(word, embeddings, tokenizer, top_k=5):
    """Βρες τα token με τις πιο κοντινές διανυσματικές αναπαραστάσεις σε εκείνη του πρώτου token της λέξης."""
    # Βρες το αναγνωριστικό του πρώτου token της λέξης
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    target_vec = embeddings[token_id]
    
    # Υπολόγισε την ομοιότητα συνημιτόνου με όλα τα token
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),  # (1, 768)
        embeddings,               # (50257, 768)
        dim=1
    )
    
    # Πάρε τα top_k πιο όμοια token (εξαιρώντας το ίδιο το token)
    top_indices = similarities.argsort(descending=True)[1:top_k+1]
    
    print(f"\nΤα token που μοιάζουν περισσότερο με το '{word}':")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Εξερεύνησε σημασιολογικές σχέσεις
find_similar_tokens("king", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("computer", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("happy", token_embeddings, tokenizer, top_k=8)

### Πειραματίσου κι εσύ!

In [ ]:
# Δοκίμασε διαφορετικές λέξεις:
words_to_explore = ["Python", "doctor", "fast", "beautiful"]

for word in words_to_explore:
    find_similar_tokens(word, token_embeddings, tokenizer, top_k=5)

### Κάτι ακόμα: Αναλογίες λέξεων

Μπορείς να βρεις διανύσματα που ικανοποιούν τη σχέση «king - man + woman ≈ queen»;

In [ ]:
def word_analogy(word1, word2, word3, embeddings, tokenizer, top_k=5):
    """
    Βρες: word1 - word2 + word3 ≈ ?
    Παράδειγμα: king - man + woman ≈ queen
    """
    # Πάρε τα αναγνωριστικά των token
    id1 = tokenizer.encode(word1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(word2, add_special_tokens=False)[0]
    id3 = tokenizer.encode(word3, add_special_tokens=False)[0]
    
    # Υπολόγισε το διάνυσμα-στόχο: word1 - word2 + word3
    target_vec = embeddings[id1] - embeddings[id2] + embeddings[id3]
    
    # Βρες τα token με τη μεγαλύτερη ομοιότητα
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),
        embeddings,
        dim=1
    )
    
    # Εξαίρεσε τις λέξεις εισόδου από τα αποτελέσματα
    similarities[id1] = -1
    similarities[id2] = -1
    similarities[id3] = -1
    
    top_indices = similarities.argsort(descending=True)[:top_k]
    
    print(f"\n{word1} - {word2} + {word3} ≈ ?")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Κλασικό παράδειγμα: king - man + woman ≈ queen
word_analogy("king", "man", "woman", token_embeddings, tokenizer, top_k=5)

# Δοκίμασε κι άλλες αναλογίες!
word_analogy("Paris", "France", "Germany", token_embeddings, tokenizer, top_k=5)

## 6. Πρακτικές ασκήσεις

### Άσκηση 1: Χειροκίνητη αναζήτηση διανυσματικών αναπαραστάσεων

```python
# Φτιάξε ένα μικρό λεξιλόγιο (10 token) με διανυσματικές αναπαραστάσεις 8 διαστάσεων
# Δημιούργησε χειροκίνητα έναν πίνακα διανυσματικών αναπαραστάσεων και βρες τις αναπαραστάσεις για τα αναγνωριστικά token [2, 5, 7]
# Εκτύπωσε τα σχήματα σε κάθε βήμα

# Βήμα 1: Δημιούργησε τον πίνακα διανυσματικών αναπαραστάσεων
vocab_size = 10
embed_dim = 8
embedding_matrix = torch.randn(???, ???)  # Συμπλήρωσε τα μεγέθη
print(f"Σχήμα πίνακα διανυσματικών αναπαραστάσεων: {embedding_matrix.shape}")

# Βήμα 2: Φτιάξε τον τανυστή με τα αναγνωριστικά των token που θα αναζητήσεις
token_ids = torch.tensor([2, 5, 7])
print(f"Αναγνωριστικά token: {token_ids}")

# Βήμα 3: Βρες τις διανυσματικές αναπαραστάσεις (υπόδειξη: χρησιμοποίησε δείκτες, όπως στο embedding_matrix[...])
embeddings = ???
print(f"Σχήμα τανυστή διανυσματικών αναπαραστάσεων: {embeddings.shape}")
```

### Άσκηση 2: Φτιάξε την κλάση TokenEmbedding από το μηδέν

In [ ]:
# Υλοποίησε την κλάση TokenEmbedding χωρίς να κοιτάξεις το παράδειγμα
# Φρόντισε να γίνεται σωστή αρχικοποίηση και να ελέγχεται το σχήμα

# ΓΡΑΨΕ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ ΕΔΩ

### Άσκηση 3: Κωδικοποίηση θέσης

In [ ]:
# Δημιούργησε μια ακολουθία αναγνωριστικών token: [10, 20, 30, 40, 50]
# Δημιούργησε τους δείκτες θέσης και χρησιμοποίησέ τους για να βρεις τα αντίστοιχα διανύσματα στο επίπεδο διανυσματικών αναπαραστάσεων θέσης
# Επιβεβαίωσε ότι στη θέση 0 αντιστοιχεί πάντα το ίδιο διάνυσμα θέσης, ανεξάρτητα από το token

# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ

### Άσκηση 4: Ολοκλήρωσε την κλάση GPT2Embeddings

In [ ]:
# Υλοποίησε ολόκληρη την κλάση GPT2Embeddings
# Δοκίμασέ τη με:
# - Μια παρτίδα 2 ακολουθιών
# - Διαφορετικά μήκη ακολουθιών (μία μήκους 5 και μία μήκους 8, με συμπλήρωση της μικρότερης)
# - Επιβεβαίωσε ότι το σχήμα της εξόδου είναι (2, 8, embed_dim)

# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ

### Άσκηση 5: Εξερεύνησε τις ομοιότητες μεταξύ token στο GPT-2

In [ ]:
# Φόρτωσε το GPT-2 και βρες token παρόμοια με τα εξής:
# - "Python" (θα πρέπει να βρεις λέξεις σχετικές με τον προγραμματισμό)
# - "doctor" (θα πρέπει να βρεις λέξεις σχετικές με την ιατρική ή τα επαγγέλματα)
# - "fast" (θα πρέπει να βρεις λέξεις σχετικές με την ταχύτητα)
#
# Ομαδοποιεί το μοντέλο τις συναφείς έννοιες;

# ΓΡΑΨΕ ΕΔΩ ΤΟΝ ΚΩΔΙΚΑ ΣΟΥ

### Άσκηση 6: Διαχείριση συσκευών

In [ ]:
# Δημιούργησε ένα στιγμιότυπο της κλάσης GPT2Embeddings
# Μετέφερέ το στη GPU (αν υπάρχει διαθέσιμη GPU)
# Πέρασε στο επίπεδο διανυσματικών αναπαραστάσεων αναγνωριστικά token που βρίσκονται αρχικά στη CPU
# Τι συμβαίνει; Διόρθωσε το πρόβλημα μεταφέροντας πρώτα τα αναγνωριστικά token στη GPU

# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ

### Άσκηση 7: Όρια στο μήκος της ακολουθίας

In [ ]:
# Δημιούργησε ένα στιγμιότυπο της κλάσης GPT2Embeddings με max_seq_len=10
# Δοκίμασε να πάρεις τις διανυσματικές αναπαραστάσεις μιας ακολουθίας μήκους 15
# Χειρίσου το σφάλμα κόβοντας την ακολουθία στο επιτρεπόμενο μήκος προτού υπολογίσεις τις διανυσματικές αναπαραστάσεις

# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ

### Άσκηση 8: Σύνδεση με το Κεφάλαιο 8

In [ ]:
# Χρησιμοποίησε το chapter8_output.jsonl από το Κεφάλαιο 8.
# Φόρτωσε μία εγγραφή και μετέτρεψε τη λίστα token_ids σε τανυστή με σχήμα (1, μήκος ακολουθίας).
# Υπολόγισε τις διανυσματικές αναπαραστάσεις με το GPT2Embeddings
# Εκτύπωσε το σχήμα σε κάθε βήμα

# Ο ΚΩΔΙΚΑΣ ΣΟΥ ΕΔΩ

## Σύνοψη κεφαλαίου

**Τι φτιάξαμε:**

1. **Διανυσματικές αναπαραστάσεις token:** Πίνακας αναζήτησης που αντιστοιχίζει τα αναγνωριστικά token σε διανύσματα που κωδικοποιούν σημασιολογική πληροφορία
2. **Διανυσματικές αναπαραστάσεις θέσης:** Διανύσματα που μαθαίνονται κατά την εκπαίδευση και κωδικοποιούν τη θέση στην ακολουθία
3. **Πλήρες επίπεδο GPT2Embeddings:** Συνδυάζει με πρόσθεση τη διανυσματική αναπαράσταση κάθε token με εκείνη της θέσης του

**Τι μάθαμε:**

- Τα αναγνωριστικά token είναι αυθαίρετοι δείκτες χωρίς εγγενές σημασιολογικό περιεχόμενο
- Οι διανυσματικές αναπαραστάσεις αντιστοιχίζουν τα αναγνωριστικά σε πυκνά διανύσματα που αποτυπώνουν σχέσεις μεταξύ των token
- Η πληροφορία θέσης είναι καθοριστική ("cat chased dog" ≠ "dog chased cat")
- Το GPT-2 χρησιμοποιεί διανυσματικές αναπαραστάσεις θέσης που μαθαίνονται κατά την εκπαίδευση (εκπαιδεύσιμες παράμετροι)
- Η ομοιότητα συνημιτόνου συγκρίνει τις κατευθύνσεις των διανυσμάτων και μπορεί να αναδείξει σημασιολογικές σχέσεις που έμαθε το μοντέλο· αντίθετες κατευθύνσεις δεν σημαίνουν απαραίτητα αντίθετες σημασίες
- Στο GPT-2, οι διανυσματικές αναπαραστάσεις των token ομαδοποιούν κυρίως παραλλαγές που διαφέρουν στην ορθογραφία ή στη χρήση πεζών και κεφαλαίων ("King", "KING")· πλουσιότερη σημασιολογική πληροφορία αναδύεται μετά τα επίπεδα προσοχής

**Στη συνέχεια:** Στο Κεφάλαιο 10 θα χρησιμοποιήσεις αυτές τις διανυσματικές αναπαραστάσεις για την αυτοπροσοχή, όπου η αναπαράσταση κάθε token εμπλουτίζεται με πληροφορίες από τα υπόλοιπα token στα συμφραζόμενά του!